In [1]:
"""
MOSI fusion model: COMBINED evaluation + testing suite (FIXED FOR KAGGLE)
========================================================================
"""

import copy
import glob
import inspect
import json
import os
import pickle
import random
import shutil
import sys
import tempfile
import time

import numpy as np
import torch
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, mean_absolute_error)

# ============================== CONFIG (FIXED FOR KAGGLE) ==============================
import os
import sys

# 1. Force audio,video modalities for the 2-modality model
os.environ["MODALITIES"] = "audio,video"

ALL_MODS = ("text", "audio", "video")
_req = os.environ.get("MODALITIES", "audio,video").split(",")
MODALITIES = tuple(m for m in ALL_MODS if m in _req)

# 2. Set auto-fallback paths (checks /kaggle/working/ first, then /kaggle/input/)
def get_valid_path(filename, default_path):
    if os.path.exists(default_path):
        return default_path
    for root, _, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
    return default_path

DATA_PATH = get_valid_path("mosi_data.pkl", "/kaggle/input/datasets/aakarsonawane/adaptive-fusion/mosi_features/mosi_data.pkl")
CKPT_PATH = get_valid_path("best_adaptive.pt", get_valid_path("best_model.pt", "/kaggle/input/datasets/aakarsonawane/adaptive-fusion/best_model.pt"))
OUT_DIR = os.environ.get("OUT_DIR", "/kaggle/working/eval_outputs")

N_BOOT = int(os.environ.get("N_BOOT", 1000))
N_DRAWS = int(os.environ.get("N_DRAWS", 5))
CPU_THREADS = int(os.environ.get("CPU_THREADS", 2))
RUN_PIPELINE_LATENCY = os.environ.get("RUN_PIPELINE_LATENCY", "1") == "1"
D_MODEL = 128
SEQ_LEN = 50
KEY = {"text": "text", "audio": "audio", "video": "vision"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, os.getcwd())

# Ensure model.py is accessible in current working directory
if not os.path.exists("model.py"):
    model_path = get_valid_path("model.py", "")
    if model_path and os.path.exists(model_path):
        shutil.copy(model_path, "./model.py")


def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


# ============================== DATA / MODEL ==============================
def load_arrays(split):
    arr = {m: np.nan_to_num(split[KEY[m]].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
           for m in MODALITIES}
    return arr, split["labels"].astype(np.float32).reshape(-1)


def build_model(dims):
    from model import AttentiveAdaptiveFusionModel
    params = inspect.signature(AttentiveAdaptiveFusionModel.__init__).parameters
    has_text = "text_dim" in params
    if has_text != ("text" in MODALITIES):
        raise RuntimeError(
            f"model.py in {os.getcwd()} {'HAS' if has_text else 'has NO'} a text branch but MODALITIES="
            f"{MODALITIES}. Re-run the model cell of the matching notebook.")
    return AttentiveAdaptiveFusionModel(d_model=D_MODEL, **{f"{m}_dim": dims[m] for m in MODALITIES})


@torch.no_grad()
def predict(model, arrays, batch_size=64, device=None):
    device = device or DEVICE
    model.eval()
    n = len(arrays[MODALITIES[0]])
    preds, gates = [], []
    for i in range(0, n, batch_size):
        batch = [torch.from_numpy(arrays[m][i:i + batch_size]).to(device) for m in MODALITIES]
        p, g = model(*batch)
        preds.append(p.squeeze(-1).cpu().numpy())
        gates.append(g.cpu().numpy())
    return np.concatenate(preds), np.concatenate(gates)


# ============================== METRICS / STATS ==============================
def metrics(pred, y):
    corr = float(np.corrcoef(pred, y)[0, 1]) if pred.std() > 0 else 0.0
    bp, bl, nz = (pred >= 0).astype(int), (y >= 0).astype(int), y != 0
    return {"MAE": float(mean_absolute_error(y, pred)), "Corr": corr,
            "Acc2": float(accuracy_score(bl, bp)),
            "Acc2_no0": float(accuracy_score(bl[nz], bp[nz])) if np.any(nz) else 0.0,
            "F1": float(f1_score(bl, bp, average="weighted")),
            "Acc7": float(accuracy_score(np.clip(np.round(y), -3, 3), np.clip(np.round(pred), -3, 3)))}


def bootstrap_metrics(pred, y, rng, B):
    n = len(y)
    idx = rng.randint(0, n, (B, n))
    runs = [metrics(pred[i], y[i]) for i in idx]
    out = {}
    for k in runs[0]:
        v = np.array([r[k] for r in runs])
        out[k] = {"mean": float(v.mean()), "lo": float(np.percentile(v, 2.5)), "hi": float(np.percentile(v, 97.5))}
    return out, idx


def holm(pvals):
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj


# ============================== A. EDA ==============================
def section_eda(train_y, valid_y, test_y, arrays_test, arrays_train_sample):
    print("\n" + "=" * 90 + "\nA. EDA\n" + "=" * 90)
    res = {"labels": {}, "features": {}}
    for name, y in (("train", train_y), ("valid", valid_y), ("test", test_y)):
        d = {"n": int(len(y)), "mean": float(y.mean()), "std": float(y.std()),
             "positive(>0)": int((y > 0).sum()), "zero(=0)": int((y == 0).sum()), "negative(<0)": int((y < 0).sum()),
             "acc2_majority_baseline": float(max((y >= 0).mean(), (y < 0).mean()))}
        res["labels"][name] = d
        print(f"{name:<6} n={d['n']:<5} mean={d['mean']:+.3f} std={d['std']:.3f} | pos {d['positive(>0)']} "
              f"zero {d['zero(=0)']} neg {d['negative(<0)']} | majority Acc-2 baseline {d['acc2_majority_baseline']:.4f}")
    for m in MODALITIES:
        x = arrays_test[m]
        zero_frames = (np.abs(x).sum(-1) == 0)
        res["features"][m] = {"dim": int(x.shape[-1]), "mean": float(x.mean()), "std": float(x.std()),
                              "min": float(x.min()), "max": float(x.max()),
                              "pct_all_zero_frames": float(100 * zero_frames.mean()),
                              "pct_samples_with_over_50pct_zero_frames": float(100 * (zero_frames.mean(1) > 0.5).mean())}
        f = res["features"][m]
        print(f"{m:<6} dim={f['dim']:<5} mean={f['mean']:+.3f} std={f['std']:.3f} range=[{f['min']:.2f},{f['max']:.2f}] "
              f"| all-zero frames {f['pct_all_zero_frames']:.1f}% | samples >50% zero frames "
              f"{f['pct_samples_with_over_50pct_zero_frames']:.1f}%")
    if "video" in MODALITIES:
        print("  (video all-zero frames = padding + frames where no face was detected)")
    plt.figure(figsize=(7, 4))
    plt.hist(train_y, bins=30, alpha=0.6, label="train", density=True)
    plt.hist(test_y, bins=30, alpha=0.6, label="test", density=True)
    plt.xlabel("sentiment label")
    plt.ylabel("density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "label_distribution.png"), dpi=150)
    plt.close()
    return res


# ============================== B. STATISTICAL EVALUATION ==============================
def section_stats(model, arrays, y, rng):
    print("\n" + "=" * 90 + "\nB. STATISTICAL EVALUATION (test split)\n" + "=" * 90)
    pred, gates = predict(model, arrays)
    m = metrics(pred, y)
    boot, idx = bootstrap_metrics(pred, y, rng, N_BOOT)
    base = max((y >= 0).mean(), (y < 0).mean())
    correct = ((pred >= 0) == (y >= 0)).astype(float)
    diff = np.array([correct[i].mean() for i in idx]) - base
    print(f"{'Metric':<10}{'Point':>9}{'95% CI':>22}")
    for k in m:
        print(f"{k:<10}{m[k]:>9.4f}   [{boot[k]['lo']:.4f}, {boot[k]['hi']:.4f}]")
    print(f"\nMajority-class Acc-2 baseline: {base:.4f} | model - baseline = {m['Acc2'] - base:+.4f} "
          f"(95% CI [{np.percentile(diff, 2.5):+.4f}, {np.percentile(diff, 97.5):+.4f}])  "
          f"-> {'significantly above' if np.percentile(diff, 2.5) > 0 else 'NOT clearly above'} baseline")
    bl, bp = (y >= 0).astype(int), (pred >= 0).astype(int)
    print("\nClassification report (0 = negative, 1 = non-negative):")
    print(classification_report(bl, bp, target_names=["negative", "non-negative"], digits=4))
    cm = confusion_matrix(bl, bp)
    print("Confusion matrix [[TN FP],[FN TP]]:\n", cm)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    ax[0].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax[0].text(j, i, str(v), ha="center", va="center")
    ax[0].set_xticks([0, 1])
    ax[0].set_yticks([0, 1])
    ax[0].set_xticklabels(["pred neg", "pred non-neg"])
    ax[0].set_yticklabels(["true neg", "true non-neg"])
    ax[0].set_title("Confusion matrix")
    ax[1].scatter(y, pred, s=8, alpha=0.5)
    ax[1].plot([-3, 3], [-3, 3], "r--")
    ax[1].set_xlabel("true")
    ax[1].set_ylabel("predicted")
    ax[1].set_title("Prediction vs label")
    ax[2].hist(pred - y, bins=30)
    ax[2].set_title("Residuals (pred - true)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "stats_figures.png"), dpi=150)
    plt.close()

    err = np.abs(pred - y)
    worst = np.argsort(-err)[:10]
    print("\nTop-10 largest errors (sample index, true, pred):")
    for i in worst:
        print(f"  #{int(i):<4} true {y[i]:+.2f}  pred {pred[i]:+.2f}  |err| {err[i]:.2f}")
    print("\nMAE by true-sentiment bin:")
    bins = [(-3.1, -2), (-2, -1), (-1, 0), (0, 1), (1, 2), (2, 3.1)]
    by_bin = {}
    for lo, hi in bins:
        sel = (y >= lo) & (y < hi)
        if sel.sum():
            by_bin[f"[{lo:.0f},{hi:.0f})"] = {"n": int(sel.sum()), "MAE": float(err[sel].mean())}
            print(f"  [{max(lo, -3):+.0f}, {min(hi, 3):+.0f})  n={int(sel.sum()):<4} MAE={err[sel].mean():.3f}")
    return {"point": m, "ci": boot, "baseline_acc2": float(base),
            "diff_vs_baseline_ci": [float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))],
            "confusion_matrix": cm.tolist(), "mae_by_bin": by_bin,
            "worst_idx": [int(i) for i in worst], "mean_gate_clean": gates.mean(0).tolist()}, pred, correct, idx


# ============================== C. ROBUSTNESS ==============================
def add_noise(x, std, scale, rng):
    valid = (np.abs(x).sum(-1) > 0)[..., None]
    return x + rng.standard_normal(x.shape, dtype=np.float32) * std * scale * valid


def element_dropout(x, p, rng):
    return x * (rng.random(x.shape, dtype=np.float32) >= p)


def frame_dropout(x, p, rng):
    return x * (rng.random(x.shape[:2], dtype=np.float32) >= p)[..., None]


def truncate(x, keep):
    y = x.copy()
    y[:, int(x.shape[1] * keep):, :] = 0
    return y


def build_conditions(stds):
    C = {}
    if "audio" in MODALITIES:
        for k in (0.15, 0.5, 1.0, 2.0):
            C[f"Audio noise {k}x"] = (lambda a, r, k=k: {**a, "audio": add_noise(a["audio"], stds["audio"], k, r)}, N_DRAWS)
        C["Audio truncated (keep 50%)"] = (lambda a, r: {**a, "audio": truncate(a["audio"], 0.5)}, 1)
        C["Audio truncated (keep 25%)"] = (lambda a, r: {**a, "audio": truncate(a["audio"], 0.25)}, 1)
    if "video" in MODALITIES:
        for p in (0.2, 0.5, 0.8):
            C[f"Video element dropout {int(p * 100)}%"] = (lambda a, r, p=p: {**a, "video": element_dropout(a["video"], p, r)}, N_DRAWS)
        for p in (0.3, 0.6):
            C[f"Video frame dropout {int(p * 100)}% (no face)"] = (lambda a, r, p=p: {**a, "video": frame_dropout(a["video"], p, r)}, N_DRAWS)
    if "audio" in MODALITIES and "video" in MODALITIES:
        C["Audio+video noise 1.0x"] = (lambda a, r: {**a, "audio": add_noise(a["audio"], stds["audio"], 1.0, r),
                                                     "video": add_noise(a["video"], stds["video"], 1.0, r)}, N_DRAWS)
    if "text" in MODALITIES:
        for p in (0.2, 0.4):
            C[f"Text word dropout {int(p * 100)}% (ASR proxy)"] = (lambda a, r, p=p: {**a, "text": frame_dropout(a["text"], p, r)}, N_DRAWS)
    for m in MODALITIES:
        C[f"{m.capitalize()} zeroed"] = (lambda a, r, m=m: {**a, m: np.zeros_like(a[m])}, 1)
    return C


def section_robustness(model, arrays, y, clean_correct, boot_idx, stds):
    print("\n" + "=" * 90 + "\nC. ROBUSTNESS (accuracy drop, paired bootstrap 95% CI)\n" + "=" * 90)
    print(f"(noise conditions averaged over {N_DRAWS} random draws; noise is added only to non-padding frames)")
    conds = build_conditions(stds)
    clean_acc = clean_correct.mean()
    rows = []
    for name, (fn, draws) in conds.items():
        corrs, gates = [], []
        for d in range(draws):
            rng = np.random.default_rng(1000 + d)
            pert = fn(arrays, rng)
            p, g = predict(model, pert)
            corrs.append(((p >= 0) == (y >= 0)).astype(float))
            gates.append(g.mean(0))
        pc = np.mean(corrs, axis=0)
        drops = np.array([clean_correct[i].mean() - pc[i].mean() for i in boot_idx]) * 100
        p_two = max(2 * min((drops <= 0).mean(), (drops >= 0).mean()), 1.0 / len(drops))
        rows.append({"name": name, "acc": float(pc.mean() * 100), "drop": float((clean_acc - pc.mean()) * 100),
                     "ci_lo": float(np.percentile(drops, 2.5)), "ci_hi": float(np.percentile(drops, 97.5)),
                     "p": float(p_two), "gate": np.mean(gates, axis=0).tolist()})
        print(f"  done: {name}")
    adj = holm(np.array([r["p"] for r in rows]))
    for r, a in zip(rows, adj):
        r["p_holm"] = float(a)
        r["detectable_ci"] = bool(r["ci_lo"] > 0 or r["ci_hi"] < 0)
        r["detectable_holm"] = bool(a < 0.05)
    print(f"\n{'Condition':<38}{'Acc%':>7}{'Drop':>8}{'95% CI':>20}{'CI excl. 0':>12}{'Holm p<.05':>12}")
    for r in rows:
        print(f"{r['name']:<38}{r['acc']:>7.2f}{r['drop']:>+8.2f}   [{r['ci_lo']:+6.2f},{r['ci_hi']:+6.2f}]"
              f"{'yes' if r['detectable_ci'] else 'no':>12}{'yes' if r['detectable_holm'] else 'no':>12}")
    print("\nMean gate weights (" + ", ".join(MODALITIES) + "):")
    for r in rows:
        print(f"  {r['name']:<38}" + "  ".join(f"{w:.3f}" for w in r["gate"]))
    fig, ax = plt.subplots(figsize=(10, 0.35 * len(rows) + 2))
    ax.barh([r["name"] for r in rows], [r["drop"] for r in rows],
            xerr=[[r["drop"] - r["ci_lo"] for r in rows], [r["ci_hi"] - r["drop"] for r in rows]], capsize=2)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("accuracy drop (points), 95% paired-bootstrap CI")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "robustness.png"), dpi=150)
    plt.close()
    return rows


# ============================== G. DEPLOYMENT HELPER ==============================
class Predictor:
    def __init__(self, model, dims, log_path=None):
        self.model = copy.deepcopy(model).cpu().eval()
        self.dims = dims
        self.log_path = log_path or os.path.join(OUT_DIR, "prediction_log.jsonl")

    def _prep(self, name, x):
        if x is None:
            raise ValueError(f"missing input '{name}'")
        x = np.asarray(x, dtype=np.float32)
        if x.ndim != 2 or x.shape[1] != self.dims[name]:
            raise ValueError(f"'{name}' must have shape (T, {self.dims[name]}), got {x.shape}")
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        if x.shape[0] >= SEQ_LEN:
            x = x[:SEQ_LEN]
        else:
            x = np.concatenate([x, np.zeros((SEQ_LEN - x.shape[0], x.shape[1]), np.float32)])
        return torch.from_numpy(x)[None]

    def __call__(self, text=None, audio=None, video=None):
        t0 = time.perf_counter()
        given = {"text": text, "audio": audio, "video": video}
        tensors = [self._prep(m, given[m]) for m in MODALITIES]
        with torch.no_grad():
            pred, gate = self.model(*tensors)
        score = float(pred.squeeze().item())
        out = {"score": score, "sentiment": "positive" if score >= 0 else "negative",
               "strength": float(min(abs(score) / 3.0, 1.0)),
               "gate_weights": {m: float(w) for m, w in zip(MODALITIES, gate[0].tolist())},
               "latency_ms": (time.perf_counter() - t0) * 1000}
        try:
            with open(self.log_path, "a") as f:
                f.write(json.dumps({"ts": time.time(), "score": score, "sentiment": out["sentiment"],
                                    "latency_ms": out["latency_ms"]}) + "\n")
        except OSError:
            pass
        return out


# ============================== D. UNIT TESTS ==============================
def section_unit_tests(model, arrays, dims, y):
    print("\n" + "=" * 90 + "\nD. UNIT TESTS\n" + "=" * 90)
    m_cpu = copy.deepcopy(model).cpu().eval()
    sample = {m: arrays[m][:8] for m in MODALITIES}
    results = []

    def run(name, fn):
        try:
            ok = bool(fn())
            msg = ""
        except Exception as e:
            ok, msg = False, f"{type(e).__name__}: {e}"
        results.append({"test": name, "passed": ok, "error": msg})
        print(f"  [{'PASS' if ok else 'FAIL'}] {name} {msg}")

    def t_shapes():
        p, g = predict(m_cpu, sample, device="cpu")
        return p.shape == (8,) and g.shape == (8, len(MODALITIES))

    def t_finite():
        p, g = predict(m_cpu, sample, device="cpu")
        return np.isfinite(p).all() and np.isfinite(g).all()

    def t_gate_sum():
        _, g = predict(m_cpu, sample, device="cpu")
        return np.allclose(g.sum(1), 1.0, atol=1e-5)

    def t_determinism():
        a, _ = predict(m_cpu, sample, device="cpu")
        b, _ = predict(m_cpu, sample, device="cpu")
        return np.allclose(a, b, atol=1e-7)

    def t_batch_vs_single():
        b, _ = predict(m_cpu, sample, device="cpu")
        s = np.array([predict(m_cpu, {m: sample[m][i:i + 1] for m in MODALITIES}, device="cpu")[0][0] for i in range(8)])
        return np.allclose(b, s, atol=1e-4)

    def t_zero_inputs():
        for zero in list(MODALITIES) + ["all"]:
            inp = {m: (np.zeros_like(sample[m]) if zero in (m, "all") else sample[m]) for m in MODALITIES}
            p, _ = predict(m_cpu, inp, device="cpu")
            if not np.isfinite(p).all():
                return False
        return True

    def t_short_long():
        for T in (1, 100):
            inp = {m: np.random.randn(2, T, dims[m]).astype(np.float32) for m in MODALITIES}
            p, _ = predict(m_cpu, inp, device="cpu")
            if p.shape != (2,) or not np.isfinite(p).all():
                return False
        return True

    def t_sanitize():
        pr = Predictor(model, dims, log_path=os.path.join(tempfile.gettempdir(), "ut_log.jsonl"))
        bad = {m: np.full((SEQ_LEN, dims[m]), np.nan, np.float32) for m in MODALITIES}
        bad[MODALITIES[0]][0, 0] = np.inf
        out = pr(**bad)
        return np.isfinite(out["score"])

    def t_wrong_dim():
        inp = {m: sample[m] for m in MODALITIES}
        inp[MODALITIES[0]] = np.concatenate([inp[MODALITIES[0]], np.zeros_like(inp[MODALITIES[0]][..., :1])], axis=-1)
        try:
            predict(m_cpu, inp, device="cpu")
        except Exception:
            return True
        return False

    def t_checkpoint_roundtrip():
        with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
            path = f.name
        torch.save(m_cpu.state_dict(), path)
        m2 = build_model(dims)
        m2.load_state_dict(torch.load(path, map_location="cpu"))
        m2.eval()
        a, _ = predict(m_cpu, sample, device="cpu")
        b, _ = predict(m2, sample, device="cpu")
        os.remove(path)
        return np.allclose(a, b, atol=1e-6)

    def t_predict_api():
        pr = Predictor(model, dims, log_path=os.path.join(tempfile.gettempdir(), "ut_log.jsonl"))
        batch, _ = predict(m_cpu, {m: sample[m][:3] for m in MODALITIES}, device="cpu")
        for i in range(3):
            out = pr(**{m: sample[m][i] for m in MODALITIES})
            if abs(out["score"] - float(batch[i])) > 1e-4 or set(out) < {"score", "sentiment", "gate_weights"}:
                return False
        return True

    def t_api_rejects_bad_input():
        pr = Predictor(model, dims, log_path=os.path.join(tempfile.gettempdir(), "ut_log.jsonl"))
        try:
            pr(**{m: np.zeros((SEQ_LEN, dims[m] + 1), np.float32) for m in MODALITIES})
        except ValueError:
            return True
        return False

    for name, fn in [("output shapes (pred, gate)", t_shapes), ("outputs finite", t_finite),
                     ("gate weights sum to 1", t_gate_sum), ("eval-mode determinism", t_determinism),
                     ("batch output == single-sample output", t_batch_vs_single),
                     ("all-zero modality / all-zero input stays finite", t_zero_inputs),
                     ("very short (T=1) and long (T=100) sequences", t_short_long),
                     ("NaN/inf sanitized by predict_api", t_sanitize),
                     ("wrong feature dimension is rejected", t_wrong_dim),
                     ("checkpoint save/load round trip", t_checkpoint_roundtrip),
                     ("predict_api == batch inference", t_predict_api),
                     ("predict_api rejects malformed input", t_api_rejects_bad_input)]:
        run(name, fn)
    print(f"\n  {sum(r['passed'] for r in results)}/{len(results)} passed")
    return results


# ============================== E. EFFICIENCY ==============================
def bench(model, arrays, device, batch, runs):
    m = copy.deepcopy(model).to(device).eval()
    x = [torch.from_numpy(arrays[mod][:batch]).to(device) for mod in MODALITIES]
    times = []
    with torch.no_grad():
        for _ in range(5):
            m(*x)
        for _ in range(runs):
            if device == "cuda":
                torch.cuda.synchronize()
            t = time.perf_counter()
            m(*x)
            if device == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t) * 1000)
    t = np.array(times)
    return {"batch": batch, "p50": float(np.percentile(t, 50)), "p95": float(np.percentile(t, 95)),
            "p99": float(np.percentile(t, 99)), "mean": float(t.mean()), "samples_per_sec": float(batch / (t.mean() / 1000))}


def section_efficiency(model, arrays):
    print("\n" + "=" * 90 + "\nE. EFFICIENCY\n" + "=" * 90)
    n_params = sum(p.numel() for p in model.parameters())
    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        path = f.name
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / 1e6
    os.remove(path)
    print(f"Parameters: {n_params:,} | checkpoint size: {size_mb:.2f} MB")
    res = {"params": int(n_params), "size_mb": float(size_mb), "cpu": [], "gpu": []}
    old = torch.get_num_threads()
    torch.set_num_threads(CPU_THREADS)
    print(f"\nForward-pass latency, CPU ({CPU_THREADS} threads):")
    for b, runs in ((1, 100), (8, 50), (32, 30), (64, 20)):
        r = bench(model, arrays, "cpu", b, runs)
        res["cpu"].append(r)
        print(f"  batch {b:<3} p50 {r['p50']:>8.2f} ms  p95 {r['p95']:>8.2f}  p99 {r['p99']:>8.2f}  {r['samples_per_sec']:>8.1f} samples/s")
    torch.set_num_threads(old)
    if torch.cuda.is_available():
        print("\nForward-pass latency, GPU:")
        for b, runs in ((1, 100), (8, 50), (32, 30), (64, 20)):
            r = bench(model, arrays, "cuda", b, runs)
            res["gpu"].append(r)
            print(f"  batch {b:<3} p50 {r['p50']:>8.2f} ms  p95 {r['p95']:>8.2f}  p99 {r['p99']:>8.2f}  {r['samples_per_sec']:>8.1f} samples/s")
    else:
        print("\n(no GPU in this session: GPU latency not measured)")
    return res


# ============================== F. END-TO-END PIPELINE LATENCY ==============================
def find_raw_root(base="/kaggle/input"):
    for root, dirs, _ in os.walk(base):
        if root[len(base):].count(os.sep) > 4:
            dirs[:] = []
            continue
        if "Raw" in dirs:
            return os.path.join(root, "Raw")
    return None


def section_pipeline_latency(model, dims, n_clips=10):
    print("\n" + "=" * 90 + "\nF. END-TO-END PIPELINE LATENCY (raw clip -> features -> model)\n" + "=" * 90)
    raw = find_raw_root()
    if raw is None or not ({"audio", "video"} <= set(MODALITIES)):
        print("Skipped (raw MOSI folder not found, or model has no audio/video).")
        return None
    import cv2
    import librosa
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    aud_dir = os.path.join(raw, "Audio", "WAV_16000", "Segmented")
    vid_dir = os.path.join(raw, "Video", "Segmented")
    clips = [p for p in sorted(glob.glob(os.path.join(vid_dir, "*.mp4")))
             if os.path.exists(os.path.join(aud_dir, os.path.basename(p)[:-4] + ".wav"))]
    if not clips:
        print("Skipped (no matching raw clips found).")
        return None

    random.Random(0).shuffle(clips)
    clips = clips[:n_clips]

    def pad(x):
        x = x[:SEQ_LEN]
        return np.concatenate([x, np.zeros((SEQ_LEN - len(x), x.shape[1]), np.float32)]) if len(x) < SEQ_LEN else x

    def audio_feats(path, hop=512):
        y, sr = librosa.load(path, sr=16000)
        yy = np.pad(y, (0, max(0, hop - len(y))))
        mfcc = librosa.feature.mfcc(y=yy, sr=sr, n_mfcc=13, hop_length=hop)
        f = np.concatenate([mfcc, librosa.feature.delta(mfcc), librosa.feature.zero_crossing_rate(y=yy, hop_length=hop),
                            librosa.feature.rms(y=yy, hop_length=hop)], axis=0)
        return f.T.astype(np.float32), len(y) / sr

    def video_feats(path, every=3, crop=48):
        cap = cv2.VideoCapture(path)
        out, i = [], 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if i % every == 0:
                g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                faces = cascade.detectMultiScale(g, 1.1, 5, minSize=(40, 40))
                if len(faces):
                    x, yv, w, h = max(faces, key=lambda b: b[2] * b[3])
                    out.append(cv2.resize(g[yv:yv + h, x:x + w], (crop, crop)).flatten().astype(np.float32) / 255.0)
                else:
                    out.append(np.zeros(crop * crop, np.float32))
            i += 1
        cap.release()
        return np.stack(out) if out else np.zeros((1, crop * crop), np.float32)

    m_cpu = copy.deepcopy(model).cpu().eval()
    torch.set_num_threads(CPU_THREADS)
    rows = []
    for c in clips:
        wav = os.path.join(aud_dir, os.path.basename(c)[:-4] + ".wav")
        t0 = time.perf_counter()
        a, dur = audio_feats(wav)
        t1 = time.perf_counter()
        v = video_feats(c)
        t2 = time.perf_counter()
        inp = {"audio": pad(a)[None], "video": pad(v)[None]}
        if "text" in MODALITIES:
            inp["text"] = np.zeros((1, SEQ_LEN, dims["text"]), np.float32)
        with torch.no_grad():
            m_cpu(*[torch.from_numpy(inp[m]) for m in MODALITIES])
        t3 = time.perf_counter()
        rows.append({"clip_sec": dur, "audio_ms": (t1 - t0) * 1000, "video_ms": (t2 - t1) * 1000,
                     "model_ms": (t3 - t2) * 1000, "total_ms": (t3 - t0) * 1000, "rtf": (t3 - t0) / max(dur, 1e-6)})
    res = {k: {"median": float(np.median([r[k] for r in rows])), "p95": float(np.percentile([r[k] for r in rows], 95))}
           for k in ("clip_sec", "audio_ms", "video_ms", "model_ms", "total_ms", "rtf")}
    print(f"{len(rows)} random raw clips, CPU ({CPU_THREADS} threads), one clip at a time:")
    for k, label in (("clip_sec", "clip duration (s)"), ("audio_ms", "audio features (ms)"),
                     ("video_ms", "face detection + video features (ms)"), ("model_ms", "model forward (ms)"),
                     ("total_ms", "TOTAL (ms)"), ("rtf", "real-time factor (total / clip duration)")):
        print(f"  {label:<42} median {res[k]['median']:>9.2f}   p95 {res[k]['p95']:>9.2f}")
    return res


# ============================== H. REPORT ==============================
def print_markdown_tables(R):
    print("\n" + "=" * 90 + "\nH. TABLES TO PASTE INTO THE REPORT\n" + "=" * 90)
    s = R["stats"]
    print("\n**Table 1. Test-set performance (95% bootstrap CI)**\n\n| Metric | Value | 95% CI |\n|---|---|---|")
    for k, v in s["point"].items():
        print(f"| {k} | {v:.4f} | [{s['ci'][k]['lo']:.4f}, {s['ci'][k]['hi']:.4f}] |")
    print(f"| Majority Acc-2 baseline | {s['baseline_acc2']:.4f} | n/a |")
    if R.get("robustness"):
        print("\n**Table 2. Robustness (accuracy drop in points, paired bootstrap CI)**\n\n"
              "| Condition | Acc (%) | Drop | 95% CI | CI excludes 0 | Holm p<0.05 |\n|---|---|---|---|---|---|")
        for r in R["robustness"]:
            print(f"| {r['name']} | {r['acc']:.2f} | {r['drop']:+.2f} | [{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}] | "
                  f"{'yes' if r['detectable_ci'] else 'no'} | {'yes' if r['detectable_holm'] else 'no'} |")
    e = R["efficiency"]
    print(f"\n**Table 3. Efficiency** (params {e['params']:,}, checkpoint {e['size_mb']:.2f} MB)\n\n"
          "| Device | Batch | p50 (ms) | p95 (ms) | p99 (ms) | Samples/s |\n|---|---|---|---|---|---|")
    for dev, key in (("CPU", "cpu"), ("GPU", "gpu")):
        for r in e[key]:
            print(f"| {dev} | {r['batch']} | {r['p50']:.2f} | {r['p95']:.2f} | {r['p99']:.2f} | {r['samples_per_sec']:.1f} |")
    t = R["unit_tests"]
    print(f"\n**Table 4. Unit tests: {sum(x['passed'] for x in t)}/{len(t)} passed**")


def main():
    set_seed(42)
    print(f"Modalities: {MODALITIES} | device: {DEVICE} | data: {DATA_PATH} | checkpoint: {CKPT_PATH}")
    with open(DATA_PATH, "rb") as f:
        data = pickle.load(f)
    test_arr, test_y = load_arrays(data["test"])

    train_split_subset = {k: v[:500] for k, v in data["train"].items()}
    train_arr, _ = load_arrays(train_split_subset)
    train_labels = data["train"]["labels"].astype(np.float32).reshape(-1)
    valid_y = data["valid"]["labels"].astype(np.float32).reshape(-1)

    dims = {m: test_arr[m].shape[-1] for m in MODALITIES}
    stds = {m: train_arr[m].reshape(-1, dims[m]).std(0).reshape(1, 1, -1).astype(np.float32) for m in MODALITIES}

    model = build_model(dims)
    model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu"))
    model.to(DEVICE).eval()

    R = {"modalities": list(MODALITIES), "eda": section_eda(train_labels, valid_y, test_y, test_arr, train_arr)}
    rng = np.random.RandomState(42)
    R["stats"], pred, clean_correct, boot_idx = section_stats(model, test_arr, test_y, rng)
    R["robustness"] = section_robustness(model, test_arr, test_y, clean_correct, boot_idx, stds)
    R["unit_tests"] = section_unit_tests(model, test_arr, dims, test_y)
    R["efficiency"] = section_efficiency(model, test_arr)
    R["pipeline_latency"] = section_pipeline_latency(model, dims) if RUN_PIPELINE_LATENCY else None
    print_markdown_tables(R)
    out = os.path.join(OUT_DIR, "eval_results.json")
    with open(out, "w") as f:
        json.dump(R, f, indent=2, default=float)
    print(f"\nSaved results -> {out}\nFigures in {OUT_DIR}: label_distribution.png, stats_figures.png, robustness.png")


if __name__ == "__main__":
    main()

Modalities: ('audio', 'video') | device: cuda | data: /kaggle/input/datasets/aakarsonawane/adaptive-fusion/mosi_features/mosi_data.pkl | checkpoint: /kaggle/input/datasets/aakarsonawane/adaptive-fusion/best_model.pt

A. EDA
train  n=1283  mean=+0.235 std=1.510 | pos 679 zero 52 neg 552 | majority Acc-2 baseline 0.5698
valid  n=229   mean=+0.310 std=1.649 | pos 124 zero 13 neg 92 | majority Acc-2 baseline 0.5983
test   n=686   mean=-0.317 std=1.588 | pos 277 zero 30 neg 379 | majority Acc-2 baseline 0.5525
audio  dim=28    mean=-12.506 std=72.357 range=[-798.82,279.35] | all-zero frames 1.8% | samples >50% zero frames 0.3%
video  dim=2304  mean=+0.271 std=0.250 range=[0.00,1.00] | all-zero frames 28.1% | samples >50% zero frames 25.9%
  (video all-zero frames = padding + frames where no face was detected)

B. STATISTICAL EVALUATION (test split)
Metric        Point                95% CI
MAE          1.6510   [1.5697, 1.7382]
Corr        -0.0840   [-0.1563, -0.0097]
Acc2         0.4854   